In [9]:
import json
import time
import pandas as pd
import serpapi
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
SERPAPI_KEY = os.getenv('SERPAPI_KEY')

OUTPUT_DIR = Path("data/raw/popular_times")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = serpapi.Client(api_key=SERPAPI_KEY)


In [10]:
DAY_MAP = {
    "monday"   : 0,
    "tuesday"  : 1,
    "wednesday": 2,
    "thursday" : 3,
    "friday"   : 4,
    "saturday" : 5,
    "sunday"   : 6,
}

VENUES = pd.read_csv("../data/raw/venue/venue_malang.csv")

# VENUES = [
#     {"name": "Alun-alun Kota Malang",      "zone": "public_area"},
#     {"name": "Taman Trunojoyo Malang",     "zone": "public_area"},
#     {"name": "Taman Adipura Malang",       "zone": "public_area"},
#     {"name": "Taman Singha Merjosari",     "zone": "public_area"}, 
#     {"name": "Hutan Kota Malabar Malang",  "zone": "public_area"},
#     {"name": "Lapangan Rampal Malang",     "zone": "public_area"},
#     {"name": "Pasar Besar Malang",         "zone": "commercial_area"},
#     {"name": "Pasar Blimbing Malang",      "zone": "commercial_area"},
#     {"name": "Perpustakaan Kota Malang",   "zone": "public_area"},
#     {"name": "Masjid Jami Kota Malang",    "zone": "worship_area"},
#     {"name": "Masjid Sabilillah Malang",   "zone": "worship_area"},
#     {"name": "Masjid Raden Patah Universitas Brawijaya Malang", "zone": "worship_area"},
#     {"name": "Masjid Al-Hikmah Universitas Negeri Malang",      "zone": "worship_area"},
#     {"name": "Masjid AR Fachruddin UMM Malang",                 "zone": "worship_area"},
#     {"name": "Masjid Agung Jami Malang",   "zone": "worship_area"},
    
#     {"name": "Taman Slamet",               "zone": "public_area"},
#     {"name": "Taman Kota Malang",          "zone": "public_area"},
#     {"name": "Taman Ijen",                 "zone": "public_area"},
#     {"name": "Pasar Terpadu Dinoyo",       "zone": "commercial_area"},
#     {"name": "Malang City Hall",           "zone": "public_area"},
#     {"name": "Kampoeng Heritage Kajoetangan",        "zone": "public_area"}
# ]

In [11]:
def get_place_results(keyword):
    results = client.search({
    "engine": "google_maps",
    "q": keyword,
    "type": "search"
    })

    return results.get('place_results', {})

def convert_to_24h(hour_str):
    parts = hour_str.split(" ")
    num = int(parts[0])
    period = parts[1].upper() 

    if period == "AM":
        if num == 12:
            return 0  
        return num   
    else:
        if num == 12:
            return 12 
        return num + 12 


In [ ]:
rows = []

for _, venue in VENUES.iterrows():
  place_results = get_place_results(venue["venue_name"])
  popular_time = place_results.get('popular_times')
  gps_coordinate = place_results.get('gps_coordinates') or {}

  if popular_time:
    graph_results = popular_time.get('graph_results')
    
    for day_name, hours_list in graph_results.items():
      for hour_data in hours_list:
        rows.append({
          "venue_name" : venue["venue_name"],
          "lat" : gps_coordinate.get('latitude'),
          "lon" : gps_coordinate.get('longitude'),
          "zone" : venue["zone"], 
          "day_name" : day_name,
          "day_int" : DAY_MAP[day_name],
          "hour" : convert_to_24h(hour_data["time"]),
          "busyness_score" : hour_data["busyness_score"]
          })

    print(f"finished collecting data: {venue['venue_name']}")
    time.sleep(2)
  
if rows:
    df_popular_times = pd.DataFrame(rows)
    df_popular_times.to_csv("../data/raw/popular_times/popular_times_malang.csv", index=False)
else:
    print("Failed to collecting data")



    




finished collecting data: Masjid Al-Falah MAN 2 Kota Malang
finished collecting data: Masjid Tarbiyah UIN Malang
finished collecting data: Masjid Quba Ijen Malang
finished collecting data: Masjid Ahmad Yani Malang
finished collecting data: Masjid Al-Ikhlas Malang
finished collecting data: Masjid Baitut Taqwa Bea Cukai Malang
finished collecting data: Masjid Al-Falah Stasiun Malang Kotabaru
finished collecting data: Masjid Nur Inka Museum Brawijaya Malang
finished collecting data: Masjid Al-Huda Embong Arab Malang
finished collecting data: Masjid Ibnu Sina Veteran Malang
finished collecting data: Masjid Jami Nurul Falah Malang
finished collecting data: Masjid Darussalam Al-Wahid Malang
finished collecting data: Masjid Darussalam PNP Tanjungrejo Malang
finished collecting data: Masjid Al-Amien Sukun Gempol Malang
finished collecting data: Masjid Al-Anshor Sukun Malang
finished collecting data: Masjid Ar-Ridho Jalan Gamalama Malang
finished collecting data: Masjid Jami Arroudloh Malang
fi